# Entanglement and the Bell state $|\Phi^+\rangle$

### Subject: Introduction to Quantum Mechanics

We build the Bell state

$$ |\Phi^+\rangle = \frac{|00\rangle + |11\rangle}{\sqrt{2}} $$

with an $H$ + CNOT circuit and measure the correlations. Individually each
qubit looks random; jointly the outcomes are **always equal** — that is
entanglement.

If Qiskit is available we use the Aer simulator; otherwise the same math is
done with NumPy.


## 0. Setup


In [1]:
import sys
from pathlib import Path

# Make quantum/_common importable whether the kernel cwd is the repo root
# or the subject folder.
_here = Path.cwd().resolve()
for candidate in (_here, _here.parent, _here.parent.parent):
    if (candidate / "_common").is_dir():
        sys.path.insert(0, str(candidate))
        break

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (8, 4.5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

from _common import ket0, HADAMARD

def qiskit_available() -> bool:
    try:
        import qiskit  # noqa: F401
        import qiskit_aer  # noqa: F401
        return True
    except ImportError:
        return False

print("Qiskit available?", qiskit_available())


Qiskit available? False


## 1. Build $|\Phi^+\rangle$ with linear algebra

1. Start from $|00\rangle$.
2. Apply $H\otimes I$ → $\frac{|00\rangle+|10\rangle}{\sqrt{2}}$.
3. Apply CNOT → $\frac{|00\rangle+|11\rangle}{\sqrt{2}}$.


In [2]:
def bell_state_numpy() -> np.ndarray:
    state = np.kron(ket0, ket0)
    state = np.kron(HADAMARD, np.eye(2)) @ state
    cnot = np.array(
        [[1, 0, 0, 0],
         [0, 1, 0, 0],
         [0, 0, 0, 1],
         [0, 0, 1, 0]],
        dtype=complex,
    )
    return cnot @ state


phi_plus = bell_state_numpy()
probs = np.abs(phi_plus.flatten()) ** 2
labels = ["00", "01", "10", "11"]
print("Amplitudes:", np.round(phi_plus.flatten(), 3))
print("Probabilities:")
for lab, p in zip(labels, probs):
    print(f"  |{lab}>: {p:.3f}")


Amplitudes: [0.707+0.j 0.   +0.j 0.   +0.j 0.707+0.j]
Probabilities:
  |00>: 0.500
  |01>: 0.000
  |10>: 0.000
  |11>: 0.500


## 2. Sample measurements (NumPy)


In [3]:
shots = 1000
outcomes = np.random.choice(labels, size=shots, p=probs)
counts_np = {lab: int(np.sum(outcomes == lab)) for lab in labels}
print(f"NumPy sampling ({shots} shots):")
for lab, n in counts_np.items():
    print(f"  |{lab}>: {n:>4}  ({100 * n / shots:.1f}%)")
same = counts_np["00"] + counts_np["11"]
print(f"\nCorrelated (00 or 11): {100 * same / shots:.1f}%  -> expected ~100%")


NumPy sampling (1000 shots):
  |00>:  526  (52.6%)
  |01>:    0  (0.0%)
  |10>:    0  (0.0%)
  |11>:  474  (47.4%)

Correlated (00 or 11): 100.0%  -> expected ~100%


## 3. Same circuit with Qiskit (if installed)

Circuit: `H` on qubit 0, then `CNOT(0→1)`, then measure both qubits.


In [4]:
if qiskit_available():
    from qiskit import QuantumCircuit
    from qiskit_aer import AerSimulator

    qc = QuantumCircuit(2, 2)
    qc.h(0)
    qc.cx(0, 1)
    qc.measure([0, 1], [0, 1])
    print(qc.draw(output="text"))

    result = AerSimulator().run(qc, shots=shots).result()
    counts_qk = dict(sorted(result.get_counts().items()))
    print(f"\nQiskit + Aer ({shots} shots):")
    for lab, n in counts_qk.items():
        print(f"  |{lab}>: {n:>4}  ({100 * n / shots:.1f}%)")
    same_qk = counts_qk.get("00", 0) + counts_qk.get("11", 0)
    print(f"\nCorrelated (00 or 11): {100 * same_qk / shots:.1f}%")
else:
    print("Qiskit not installed — NumPy result above is enough.")
    print("Install with: pip install qiskit qiskit-aer")


Qiskit not installed — NumPy result above is enough.
Install with: pip install qiskit qiskit-aer


### Next steps
- Bell's inequality / CHSH test.
- The four Bell states and how to distinguish them.
- Quantum teleportation with 3 qubits.
